# Pairwise evaluation with a Kaggle judge model

Reads the blinded pair files from the generation and LoRA notebooks, asks a Kaggle model-proxy LLM which response is better, maps each A/B verdict back to the models **after** judging, and summarizes wins, losses, ties, unknowns and failures.

**Before running**
1. Push the repo (`src/alignment/judging.py`, `scripts/judge_pairwise_kaggle.py`).
2. Set this notebook up as a **utility benchmark** notebook with model access enabled (manual §13.1). That's what gives `kaggle_benchmarks` free access to judge models. No GPU needed. **Internet = On**.
3. **Add Input** → the saved **generation** notebook version (03) and the saved **LoRA** notebook version (04). Their `outputs/eval/*_pairs.jsonl` files are found automatically.
4. Private repo: Kaggle secret `GITHUB_TOKEN`.
5. **Resuming:** if a run stops partway, save the version, attach it as an input and run again. Finished judgments are copied in and skipped (`--reuse_judgments`), and failed batches are retried.

**How to use**
- **Pilot:** `MODE = "pilot"` (10 pairs of the first file, about 1 minute). Read the printed raw judge reply and verdicts.
- **Full:** `MODE = "full"`, then **Save & Run All**. Call budget at 5 pairs per call:
  - required comparisons + persona, both A/B orders: 2 × 600 + 80 judgments = **256 calls**
  - three optional comparisons, one order: 900 judgments = **180 calls**

  At about 10–15 s per call (5 s polite sleep + model time), that's **about 1.5–2 h**. To halve it, drop the optional rows from `COMPARISONS`.

In [ ]:
# ---- Configuration ----
MODE = "pilot"                        # "pilot" or "full"

REPO_URL = "https://github.com/HafsaKamran/small-model-alignment-starter.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/small-model-alignment"
EVAL_DIR = f"{REPO_DIR}/outputs/eval"                      # pair files are copied here
JUDGED_DIR = f"{REPO_DIR}/outputs/" + ("judged_pilot" if MODE == "pilot" else "judged")

JUDGE_MODEL = "google/gemini-3.6-flash"   # manual's suggestion; the next cell lists alternatives if it's missing
JUDGE_ARGS = ["--batch_size", "5", "--sleep_seconds", "5", "--max_retries", "3", "--reuse_judgments"]

# (pair file stem, judge both A/B orders?). Required comparisons first.
# Both orders doubles the calls but measures and removes position bias, so it's on for the headline results.
BOTH_ORDERS_FOR_OPTIONAL = False
COMPARISONS = [
    ("alpaca_dpo_ultrafeedback_vs_sft", True),            # required: UltraFeedback-DPO vs SFT on AlpacaEval
    ("hh_dpo_hh_vs_sft", True),                           # required: HH-DPO vs SFT on the HH eval set
    ("persona_lora_vs_base", True),                       # Task 5: LoRA-DPO vs base SmolLM2 (40 pairs)
    ("alpaca_dpo_hh_vs_sft", BOTH_ORDERS_FOR_OPTIONAL),   # optional: alignment tax of HH-DPO on helpfulness
    ("hh_dpo_ultrafeedback_vs_sft", BOTH_ORDERS_FOR_OPTIONAL),  # optional: UF-DPO on safety prompts
    ("alpaca_sft_vs_base", BOTH_ORDERS_FOR_OPTIONAL),     # optional: what SFT bought over the base model
]
if MODE == "pilot":
    COMPARISONS, JUDGE_ARGS = COMPARISONS[:1], JUDGE_ARGS + ["--limit", "10"]
assert MODE in ("pilot", "full")
print(MODE, JUDGED_DIR, JUDGE_MODEL)

## 1. Repo and judge connection

In [ ]:
import os, subprocess, sys

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
    print("Using GITHUB_TOKEN secret")
except Exception:
    print("No GITHUB_TOKEN secret; cloning as a public repo")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log -1 --oneline
sys.path.insert(0, REPO_DIR)
!PYTHONPATH=. python -m pytest -q tests/test_judging.py

One real call before the batch job: confirms model access works and shows which judge models are available. If `JUDGE_MODEL` isn't in the list, pick another from it and **record your choice in the report**.

In [ ]:
import kaggle_benchmarks as kbench
from src.alignment.judging import KaggleJudge

available = sorted(kbench.llms.keys())
print(f"{len(available)} models available:", available)
judge = KaggleJudge(JUDGE_MODEL)      # raises with the list above if the name doesn't match
print("ping ->", judge("Reply with exactly the word OK and nothing else.")[:200])

## 2. Collect pair files (and earlier judgments, when resuming)

In [ ]:
import glob, shutil

os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(JUDGED_DIR, exist_ok=True)
for path in glob.glob("/kaggle/input/**/outputs/eval/*_pairs.jsonl", recursive=True):
    shutil.copy(path, EVAL_DIR)
# Resume: judgments from an attached earlier version of this notebook
previous = [p for p in glob.glob(f"/kaggle/input/**/outputs/{os.path.basename(JUDGED_DIR)}/*", recursive=True)
            if os.path.isfile(p)]
for path in previous:
    shutil.copy(path, JUDGED_DIR)
print(f"resumed {len(previous)} files from an earlier run" if previous else "no earlier judgments attached")

missing = [stem for stem, _ in COMPARISONS if not os.path.exists(f"{EVAL_DIR}/{stem}_pairs.jsonl")]
for stem in sorted(os.listdir(EVAL_DIR)):
    print(" ", stem)
assert not missing, f"missing pair files {missing}: attach the generation (03) and LoRA (04) notebook outputs"

## 3. Judge

Each comparison runs in turn. Every batch is written to disk as soon as it's judged, so an interruption loses at most one batch. A comparison that crashes is reported and skipped; rerun the cell to retry it.

For each file you get:
- `<name>_judged.jsonl`: one row per judgment. It has the raw A/B verdict, the outcome for the candidate (win/loss/tie/unknown) and the judge's one-sentence reason.
- `<name>_judged.raw.jsonl`: every raw judge reply, for auditing.
- `<name>_judged.failures.jsonl`: batches that failed every retry.
- `<name>_judged.summary.json`: the numbers for the report.

In [ ]:
from src.alignment import judging

summaries, crashed = {}, []
for stem, both_orders in COMPARISONS:
    argv = ["--pairs_path", f"{EVAL_DIR}/{stem}_pairs.jsonl", "--output_path", f"{JUDGED_DIR}/{stem}_judged.jsonl",
            "--judge_model", JUDGE_MODEL, *JUDGE_ARGS] + (["--both_orders"] if both_orders else [])
    print("\n" + "=" * 100 + f"\n{stem}  (both orders: {both_orders})", flush=True)
    try:
        summaries[stem] = judging.main(argv, judge=judge)
    except Exception as e:
        crashed.append(stem)
        print(f"!! {stem} crashed: {type(e).__name__}: {e}. Rerun this cell to resume it.", flush=True)
print("\ncrashed:", crashed or "none")

In [ ]:
# Pilot: look at what the judge actually returned
if MODE == "pilot":
    stem = COMPARISONS[0][0]
    raw = judging.load_jsonl(f"{JUDGED_DIR}/{stem}_judged.raw.jsonl")
    print("first raw reply:\n", raw[0]["reply"][:2000] if raw else "<none>")
    for row in judging.load_jsonl(f"{JUDGED_DIR}/{stem}_judged.jsonl")[:10]:
        print(f"{row['pair_id']}: judge said {row['verdicts']} -> candidate {row['outcomes']} | {row['reason']}")

## 4. Results

In [ ]:
import pandas as pd

rows = []
for stem, s in summaries.items():
    for field, r in s["fields"].items():
        o = r["overall"]
        ci = o["win_rate_no_ties_95ci"]
        rows.append({
            "comparison": stem, "criterion": field, "n": o["n_pairs"], "wins": o["wins"], "losses": o["losses"],
            "ties": o["ties"], "unknown": o["unknown"], "failures": o["failures"],
            "win rate (ties=1/2)": o["win_rate_ties_half"], "win rate (no ties)": o["win_rate_no_ties"],
            "95% CI (no ties)": f"{ci[0]:.2f}-{ci[1]:.2f}" if ci else "",
            "position consistency": r.get("position_consistency"),
            "cand words": r["length"]["candidate_mean_words"], "base words": r["length"]["baseline_mean_words"],
            "win rate when longer": r["length"]["when_candidate_longer"]["win_rate_no_ties"],
            "win rate when not longer": r["length"]["when_candidate_not_longer"]["win_rate_no_ties"],
        })
results = pd.DataFrame(rows)
results.to_csv(f"{JUDGED_DIR}/results_table.csv", index=False)
pd.set_option("display.width", 250, "display.max_columns", 30)
results

In [ ]:
# Per-category breakdown (HH categories, persona categories, AlpacaEval sources are not categories)
for stem, s in summaries.items():
    for field, r in s["fields"].items():
        if "by_category" not in r:
            continue
        print(f"\n{stem} [{field}]")
        for cat, c in r["by_category"].items():
            print(f"  {cat:32} W/L/T {c['wins']:>3}/{c['losses']:>3}/{c['ties']:>3}   win rate (ties=1/2) {c['win_rate_ties_half']}")

In [ ]:
import matplotlib.pyplot as plt

plot = results[results["n"] > 0].reset_index(drop=True)
labels = [f"{c}\n[{k}]" if k != "winner" else c for c, k in zip(plot["comparison"], plot["criterion"])]
fig, ax = plt.subplots(figsize=(10, 0.55 * len(plot) + 1.5))
left = [0] * len(plot)
for col, color in [("wins", "#2e7d32"), ("ties", "#9e9e9e"), ("losses", "#c62828"), ("unknown", "#ffb300")]:
    share = plot[col] / plot["n"]
    ax.barh(labels, share, left=left, color=color, label=col)
    left = [l + s for l, s in zip(left, share)]
ax.axvline(0.5, ls=":", c="black")
ax.set_xlabel("share of pairs (candidate's perspective)"); ax.set_xlim(0, 1); ax.invert_yaxis()
ax.legend(loc="lower right", ncol=4); ax.set_title(f"Pairwise judgments, judge = {JUDGE_MODEL}")
fig.tight_layout(); fig.savefig(f"{JUDGED_DIR}/win_rates.png", dpi=150); plt.show()

## 5. Examples for the error analysis (§14.6)

A few wins and losses with the judge's reasons, so you can check whether the judge's reasoning holds up and pick representative cases for the report.

In [ ]:
import textwrap

def show(stem, wanted, k=2, field=None):
    judged = judging.load_jsonl(f"{JUDGED_DIR}/{stem}_judged.jsonl")
    pairs = {p["pair_id"]: p for p in judging.load_jsonl(f"{EVAL_DIR}/{stem}_pairs.jsonl")}
    shown = 0
    for row in judged:
        field = field or next(iter(row["outcomes"]))
        if row["order"] != "original" or row["outcomes"][field] != wanted:
            continue
        p = pairs[row["pair_id"]]
        cand = "A" if p["hidden"]["model_a"] == p["hidden"]["candidate"] else "B"
        print("-" * 100 + f"\n[{stem}] {wanted.upper()} for candidate ({field}) | category {p.get('category')}")
        print("PROMPT:", textwrap.shorten(p["instruction"], 300))
        print(f"CANDIDATE ({cand}):", textwrap.shorten(p[f"response_{cand.lower()}"], 500))
        print(f"BASELINE:", textwrap.shorten(p["response_b" if cand == "A" else "response_a"], 500))
        print("JUDGE:", row["reason"])
        shown += 1
        if shown == k:
            break

if MODE == "full":
    for stem in ["alpaca_dpo_ultrafeedback_vs_sft", "hh_dpo_hh_vs_sft"]:
        if stem in summaries:
            show(stem, "win"); show(stem, "loss")
    if "persona_lora_vs_base" in summaries:
        show("persona_lora_vs_base", "loss", field="safety")

## 6. Package outputs

In [ ]:
import zipfile

zip_path = f"/kaggle/working/{os.path.basename(JUDGED_DIR)}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in glob.glob(f"{JUDGED_DIR}/*"):
        zf.write(path, os.path.relpath(path, f"{REPO_DIR}/outputs"))
print("wrote", zip_path, f"{os.path.getsize(zip_path)/1e6:.1f} MB")
!ls -la {JUDGED_DIR}